## Avaliando baselines e LookAhead com a ResNet-18 no dataset CIFAR-10 
*Trabalho avaliativo da disciplina de Introdução ao Aprendizado Profundo*

*Alunos: Samuel Magalhães Pereira e Marcelo Albuquerque Heluy*

In [ ]:
# Função para "desempacotar" os batches 
import matplotlib.pyplot as plt
import numpy as np

def unpickle(file):
    import pickle
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

# Desempacotando os batches

# Salvando os batches de treino
batch1 = unpickle("/kaggle/input/datasets/samuelmp07/cifar-10-python3/data_batch_1")
batch2 = unpickle("/kaggle/input/datasets/samuelmp07/cifar-10-python3/data_batch_2")
batch3 = unpickle("/kaggle/input/datasets/samuelmp07/cifar-10-python3/data_batch_3")
batch4 = unpickle("/kaggle/input/datasets/samuelmp07/cifar-10-python3/data_batch_4")
batch5 = unpickle("/kaggle/input/datasets/samuelmp07/cifar-10-python3/data_batch_5")

# Salvando batche de teste
batch_test = unpickle("/kaggle/input/datasets/samuelmp07/cifar-10-python3/test_batch")

# Preparando os dados

In [ ]:
# Preparando os dados de treinamento

X_train = np.concatenate([
    batch1[b'data'], 
    batch2[b'data'], 
    batch3[b'data'], 
    batch4[b'data'], 
    batch5[b'data']
])

# Juntando os rótulos (Listas Python)

# Como os rótulos vêm como listas normais do Python, o operador "+" junta elas.
Y_train_lista = (
    batch1[b'labels'] + 
    batch2[b'labels'] + 
    batch3[b'labels'] + 
    batch4[b'labels'] + 
    batch5[b'labels']
)

# Convertendo os rótulos para Numpy Array, melhor para o treinamento
Y_train = np.array(Y_train_lista)

print(f"Tamanho total dos dados de TREINAMENTO: {X_train.shape}")   
print(f"Tamanho total dos rótulos de TREINAMENTO: {Y_train.shape}") 

# Preparando os dados de teste

X_test = batch_test[b'data']

Y_test = np.array(batch_test[b'labels'])

print(f"Tamanho dos dados de TESTE: {X_test.shape}") 
print(f"Tamanho dos rótulos de TESTE: {Y_test.shape}")

# Visualização do Dataset

In [ ]:
# Visualizando algumas imagens para entender o dataset

for i in range(5):
    img = X_test[i] 
    img_formatada = img.reshape(3, 32, 32).transpose(1, 2, 0)
    plt.imshow(img_formatada)
    plt.title(f"Classe/Rótulo numérico: {Y_test[i]}")
    plt.axis('off')
    plt.show()

# Configurações do treinamento

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torchvision.models as models
import torch.nn.functional as F
import time

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
seed = 2024 # Definição da seed
epocas = 10 # Definindo o número de épocas de treinamento

# Preparando os dados para serem utilizados

X_train_tensor = torch.tensor(X_train, dtype=torch.float32).reshape(-1, 3, 32, 32)
X_train_tensor /= 255.0 # Para deixar entre 0 e 1 

# Converte os rótulos para Tensor de números inteiros (long)
Y_train_tensor = torch.tensor(Y_train, dtype=torch.long)

dataset = TensorDataset(X_train_tensor, Y_train_tensor) # Montando o dataset de treino
dataloader = DataLoader(dataset, batch_size=64, shuffle=True) # Como o dataset será lido

# Preparando os dados de teste
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).reshape(-1, 3, 32, 32) / 255.0
Y_test_tensor = torch.tensor(Y_test, dtype=torch.long)

# Criando lotes de 64 imagens
test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Usar placa de vídeo se disponível
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")

criterio = nn.CrossEntropyLoss() # Função de loss padrão para classificação

# Treinando o modelo e avaliando.

Marcelo: Refatorei essa célula abaixo para tornar uma função. Cada iteração da lista de otimizadores deve chamar a função model_train individualmente

In [ ]:
# Treinamento do modelo
def model_train(epocas, otimizador_nome:str='SGD'):
    # Define a semente para confiabilidade nas comparações
    torch.manual_seed(seed)

    # Garanta a semente na GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Utilizando a resnet 18 Pré treinada
    modelo = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    modelo.fc = nn.Linear(modelo.fc.in_features, 10) # Definindo 10 ao fim pois o cifar-10 tem 10 classes
    modelo = modelo.to(dispositivo)

    if otimizador_nome == 'SGD':
        otimizador = torch.optim.SGD(modelo.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
    elif otimizador_nome == 'Adam':
        otimizador = torch.optim.Adam(modelo.parameters(), lr=0.001)
    elif otimizador_nome == 'Adagrad':
        otimizador = torch.optim.Adagrad(modelo.parameters(), lr=0.01)
    elif otimizador_nome == 'RMSProp':
        otimizador = torch.optim.RMSprop(modelo.parameters(), lr=0.001, alpha=0.99)
    elif otimizador_nome == 'Lookahead':
        otimizador = Lookahead

    historico_loss = []
    tempo_inicio_total = time.time()

    print(f"\nIniciando treinamento com o otimizador: {otimizador_nome}")

    for epoca in range(epocas):
        modelo.train()
        loss_total = 0.0

        for i, (imagens, rotulos) in enumerate(dataloader):
            imagens = imagens.to(dispositivo)
            rotulos = rotulos.to(dispositivo)
            imagens_grandes = F.interpolate(imagens, size=(224, 224), mode='bilinear', align_corners=False)

            otimizador.zero_grad()
            saidas = modelo(imagens_grandes)
            loss = criterio(saidas, rotulos)

            loss.backward()
            otimizador.step()
            loss_total += loss.item()

        loss_media_epoca = loss_total / len(dataloader)
        historico_loss.append(loss_media_epoca)
        print(f"   Época [{epoca+1}/{epocas}] | Loss média: {loss_media_epoca:.4f}")

    tempo_fim_total = time.time()
    tempo_total_segundos = tempo_fim_total - tempo_inicio_total

    # Testando o modelo
    modelo.eval()
    acertos = 0
    total = 0
    with torch.no_grad():
        for imagens, rotulos in test_dataloader:
            imagens = imagens.to(dispositivo)
            rotulos = rotulos.to(dispositivo)
            imagens_grandes = F.interpolate(imagens, size=(224, 224), mode='bilinear', align_corners=False)

            saidas = modelo(imagens_grandes)
            _, previsoes = torch.max(saidas.data, 1)
            total += rotulos.size(0)
            acertos += (previsoes == rotulos).sum().item()

    acuracia = 100 * acertos / total
    print(f"   Acurácia final ({otimizador_nome}): {acuracia:.2f}%")
    torch.save(modelo.state_dict(), f'/kaggle/working/modelo_baseline_{otimizador_nome}.pth')
    return historico_loss, acuracia, tempo_total_segundos

In [ ]:
# Armazendo resultados para gerar gráfico
resultados_loss = {}
resultados_acuracia = {}
resultados_tempo = {}

otimizadores_nomes = ['SGD', 'Adam', 'Adagrad', 'RMSProp'] 

for nome in otimizadores_nomes:
    # Captura os 3 retornos da função
    loss_hist, acc, tempo = model_train(epocas, otimizador_nome=nome)
    
    # Salva no dicionário usando o nome do otimizador como chave
    resultados_loss[nome] = loss_hist
    resultados_acuracia[nome] = acc
    resultados_tempo[nome] = tempo

# **Implementação do Wrapper Lookahead**

In [ ]:
from collections import defaultdict
import torch
from torch.optim import Optimizer

In [ ]:
import torch
from torch.optim import Optimizer
from collections import defaultdict

class Lookahead(Optimizer):
    def __init__(self, otimizador_base, passos:int=5, taxa_interpolacao:float=0.5, pullback_momentum:str='none'):
        if not 0.0 <= taxa_interpolacao <= 1.0: # 0<alpha<1
            raise ValueError(f"Inválida taxa de interpolação: {taxa_interpolacao}")
        if not passos >= 1:                     # k > 1
            raise ValueError(f"Inválido número de passos: {passos}")
        if not pullback_momentum in ['reset', 'pullback', 'none']:
            raise ValueError(f"pullback_momentum deve ser 'reset', 'pullback' ou 'none'")
        
        self.otimizador_base = otimizador_base
        self.passos = passos                       # número de passos lookahead
        self.total_passos = passos
        self.taxa_interpolacao = taxa_interpolacao # fator de interpolacao.
        self.pullback_momentum = pullback_momentum # Ajusta o histórico de momentum
        
        self.state = defaultdict(dict)

        for group in self.otimizador_base.param_groups:
            for p in group['params']:
                param_state = self.state[p]
                param_state['cached_params'] = torch.zeros_like(p.data)
                param_state['cached_params'].copy_(p.data)
                if self.pullback_momentum == 'pullback':
                    param_state['cached_mom'] = torch.zeros_like(p.data)
                    
    def zero_grad(self):
        self.otimizador_base.zero_grad()

    def load_state_dict(self, state_dict):
        self.otimizador_base.load_state_dict(state_dict)

    def backup_and_load_weighs(self):
        """Para avaliação de pesos lentos"""
        for group in self.otimizador_base.param_groups: 
            for p in group['params']:
                param_state = self.state[p]
                param_state['backup_params'] = torch.zeros_like(p.data)
                param_state['backup_params'].copy_(p.data)
                p.data.copy_(param_state['cached_params'])

    def clear_and_load_backup(self):
        for group in self.otimizador_base.param_groups:
            for p in group['params']:
                param_state = self.state[p]
                p.data.copy_(param_state['backup_params'])

    def step(self, closure=None):
        """
        Realiza único passo de otimização lookahead
        closure (optional): closure que re-avalia o modelo e retorna a loss
        """
        loss = self.otimizador_base.step(closure)
        self.passos += 1
        if self.passos >= self.total_passos:
            self.passos = 0

            for group in self.otimizador_base.param_groups:
                for p in group['params']:
                    param_state = self.state[p]
                    # \theta_{lento} + \alpha (\theta_{rapido} - \theta_{lento})
                    p.data.mul_(self.taxa_interpolacao).add_(param_state['cached_params'], alpha=1.0 - self.taxa_interpolacao) 
                    param_state['cached_params'].copy_(p.data)
                    
                    if self.pullback_momentum == 'pullback':
                        # momentum_buffer só existe no SGD com momentum; outros otimizadores usam outras chaves de estado
                        if 'momentum_buffer' in self.otimizador_base.state[p]:
                            internal_momentum = self.otimizador_base.state[p]['momentum_buffer']
                            internal_momentum.mul_(self.taxa_interpolacao)
                            internal_momentum.add_(param_state['cached_mom'], alpha=1.0 - self.taxa_interpolacao)
                            param_state['cached_mom'].copy_(internal_momentum)
                    elif self.pullback_momentum == 'reset':
                        if 'momentum_buffer' in self.otimizador_base.state[p]:
                            self.otimizador_base.state[p]['momentum_buffer'] = torch.zeros_like(p.data)
                        
        return loss

# **Treinamento e Teste do LookAhead** 

In [ ]:
# Treinamento do modelo (Lookahead) — refatorado em função para testar
# as diferentes opções de pullback_momentum: 'none', 'reset', 'pullback'
def model_train_lookahead(epocas, otimizador_nome:str='SGD', pullback_momentum:str='none'):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    modelo = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    modelo.fc = nn.Linear(modelo.fc.in_features, 10)
    modelo = modelo.to(dispositivo)

    if otimizador_nome == 'SGD':
        otimizador_base = torch.optim.SGD(modelo.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
    elif otimizador_nome == 'Adam':
        otimizador_base = torch.optim.Adam(modelo.parameters(), lr=0.001)
    elif otimizador_nome == 'Adagrad':
        otimizador_base = torch.optim.Adagrad(modelo.parameters(), lr=0.01)
    elif otimizador_nome == 'RMSProp':
        otimizador_base = torch.optim.RMSprop(modelo.parameters(), lr=0.001, alpha=0.99)

    otimizador = Lookahead(otimizador_base=otimizador_base, pullback_momentum=pullback_momentum)

    historico_loss = []
    tempo_inicio_total = time.time()

    print(f"\nIniciando treinamento com Lookahead (pullback_momentum='{pullback_momentum}') | otimizador base: {otimizador_nome}")

    for epoca in range(epocas):
        modelo.train()
        loss_total = 0.0

        for i, (imagens, rotulos) in enumerate(dataloader):
            imagens = imagens.to(dispositivo)
            rotulos = rotulos.to(dispositivo)
            imagens_grandes = F.interpolate(imagens, size=(224, 224), mode='bilinear', align_corners=False)

            otimizador.zero_grad()
            saidas = modelo(imagens_grandes)
            loss = criterio(saidas, rotulos)

            loss.backward()
            otimizador.step()
            loss_total += loss.item()

        loss_media_epoca = loss_total / len(dataloader)
        historico_loss.append(loss_media_epoca)
        print(f"   Época [{epoca+1}/{epocas}] | Loss média: {loss_media_epoca:.4f}")

    tempo_fim_total = time.time()
    tempo_total_segundos = tempo_fim_total - tempo_inicio_total

    modelo.eval()
    acertos = 0
    total = 0
    with torch.no_grad():
        for imagens, rotulos in test_dataloader:
            imagens = imagens.to(dispositivo)
            rotulos = rotulos.to(dispositivo)
            imagens_grandes = F.interpolate(imagens, size=(224, 224), mode='bilinear', align_corners=False)
            saidas = modelo(imagens_grandes)
            _, previsoes = torch.max(saidas.data, 1)
            total += rotulos.size(0)
            acertos += (previsoes == rotulos).sum().item()

    acuracia = 100 * acertos / total
    print(f"   Acurácia final (Lookahead-{otimizador_nome}, pullback_momentum='{pullback_momentum}'): {acuracia:.2f}%")
    torch.save(modelo.state_dict(), f'/kaggle/working/modelo_lookahead_{otimizador_nome}_{pullback_momentum}.pth')
    return historico_loss, acuracia, tempo_total_segundos

In [ ]:
resultados_Lookahead_loss = {}
resultados_Lookahead_acuracia = {}
resultados_Lookahead_tempo = {}

otimizadores_base_nomes = ['SGD', 'Adam', 'Adagrad', 'RMSProp'] # Otimizadores que iremos avaliar
opcoes_pullback_momentum = ['none', 'reset', 'pullback'] # As 3 possibilidades da variável

# Só faz sentido testar reset/pullback pro SGD (único com momentum_buffer de verdade)
opcoes_validas_por_otimizador = {
    'SGD': ['none', 'reset', 'pullback'],
    'Adam': ['none'],
    'Adagrad': ['none'],
    'RMSProp': ['none'],
}

for pullback_momentum in opcoes_pullback_momentum:
    resultados_Lookahead_loss[pullback_momentum] = {}
    resultados_Lookahead_acuracia[pullback_momentum] = {}
    resultados_Lookahead_tempo[pullback_momentum] = {}

for nome in otimizadores_base_nomes:
    for pullback_momentum in opcoes_validas_por_otimizador[nome]:
        loss_hist, acc, tempo = model_train_lookahead(epocas, otimizador_nome=nome, pullback_momentum=pullback_momentum)

        resultados_Lookahead_loss[pullback_momentum][nome] = loss_hist
        resultados_Lookahead_acuracia[pullback_momentum][nome] = acc
        resultados_Lookahead_tempo[pullback_momentum][nome] = tempo


# Comparação entre as opções de pullback_momentum


In [ ]:
import pandas as pd

df_comparacao = pd.DataFrame(resultados_Lookahead_acuracia).round(2)
df_comparacao.index.name = "Otimizador base"
df_comparacao.columns.name = "pullback_momentum"

for nome in otimizadores_base_nomes:
    for pb in opcoes_pullback_momentum:
        if pb not in opcoes_validas_por_otimizador[nome]:
            df_comparacao.loc[nome, pb] = float('nan')

df_comparacao = df_comparacao.fillna("N/A (não se aplica)")

print("Acurácia final (%) por otimizador base e opção de pullback_momentum:\n")
display(df_comparacao)


# *Gráficos Unificados Baselines e Lookahead*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

otimizadores = ['SGD', 'Adam', 'Adagrad', 'RMSProp']

# Otimizador base e sua versão Lookahead compartilham a mesma cor
cores_loss = {
    'SGD': '#1f77b4',      
    'Adam': '#ff7f0e',     
    'Adagrad': '#2ca02c',  
    'RMSProp': '#d62728'  
}

# Baselines (linha contínua, marcador 'o')
for nome in otimizadores:
    if nome in resultados_loss:
        ax1.plot(range(1, epocas + 1), resultados_loss[nome],
                 marker='o', linestyle='-', label=f'{nome} (Baseline)',
                 color=cores_loss[nome], linewidth=2)

# Lookahead (pullback_momentum='none') todos os otimizadores (linha tracejada, marcador 'X')
for nome in otimizadores:
    if nome in resultados_Lookahead_loss['none']:
        ax1.plot(range(1, epocas + 1), resultados_Lookahead_loss['none'][nome],
                 marker='X', linestyle='--', label=f'{nome} (Lookahead)',
                 color=cores_loss[nome], linewidth=2)

# Lookahead (reset e pullback) apenas para o SGD
if 'SGD' in resultados_Lookahead_loss.get('reset', {}):
    ax1.plot(range(1, epocas + 1), resultados_Lookahead_loss['reset']['SGD'],
              marker='s', linestyle='-.', label='SGD (Lookahead-reset)',
              color=cores_loss['SGD'], linewidth=2, alpha=0.85)

if 'SGD' in resultados_Lookahead_loss.get('pullback', {}):
    ax1.plot(range(1, epocas + 1), resultados_Lookahead_loss['pullback']['SGD'],
              marker='^', linestyle=':', label='SGD (Lookahead-pullback)',
              color=cores_loss['SGD'], linewidth=2, alpha=0.85)

ax1.set_title('Convergência da Perda: Baseline vs Lookahead', fontsize=12, fontweight='bold')
ax1.set_xlabel('Épocas', fontsize=11)
ax1.set_ylabel('Perda (Loss)', fontsize=11)
ax1.set_xticks(range(1, epocas + 1))
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='best', fontsize=8)


x = np.arange(len(otimizadores))  # Localização dos grupos no eixo X
largura = 0.18                    # Largura de cada barra individual

cores_barras = {
    'Baseline': '#7f7f7f',              
    'Lookahead': '#1f77b4',            
    'Lookahead (reset)': '#ff7f0e',    
    'Lookahead (pullback)': '#2ca02c', 
}

# Para a maioria dos otimizadores temos 2 barras (Baseline, Lookahead);
# para o SGD temos 4 (Baseline, Lookahead, Lookahead-reset, Lookahead-pullback)
labels_usados = set()

for j, nome in enumerate(otimizadores):
    if nome == 'SGD':
        categorias = ['Baseline', 'Lookahead', 'Lookahead (reset)', 'Lookahead (pullback)']
        valores = [
            resultados_acuracia.get(nome, 0),
            resultados_Lookahead_acuracia['none'].get(nome, 0),
            resultados_Lookahead_acuracia.get('reset', {}).get(nome, 0),
            resultados_Lookahead_acuracia.get('pullback', {}).get(nome, 0),
        ]
    else:
        categorias = ['Baseline', 'Lookahead']
        valores = [
            resultados_acuracia.get(nome, 0),
            resultados_Lookahead_acuracia['none'].get(nome, 0),
        ]

    n = len(categorias)
    offsets = [(i - (n - 1) / 2) * largura for i in range(n)]

    for offset, categoria, valor in zip(offsets, categorias, valores):
        label = categoria if categoria not in labels_usados else None
        barra = ax2.bar(x[j] + offset, valor, largura, label=label,
                         color=cores_barras[categoria], alpha=0.9)
        labels_usados.add(categoria)

        altura = valor
        if altura > 0:
            ax2.annotate(f'{altura:.1f}%',
                         xy=(x[j] + offset, altura),
                         xytext=(0, 3),
                         textcoords="offset points",
                         ha='center', va='bottom', fontsize=7, fontweight='bold')

ax2.set_title('Acurácia Final no Dataset de Teste', fontsize=12, fontweight='bold')
ax2.set_ylabel('Acurácia (%)', fontsize=11)
ax2.set_xticks(x)
ax2.set_xticklabels(otimizadores, fontsize=11)
ax2.set_ylim(0, 110)  # Limite maior para não cortar o texto do topo
ax2.grid(True, linestyle='--', alpha=0.3, axis='y')
ax2.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9, frameon=True)

# Ajustes finais de layout e salvamento
plt.tight_layout()
plt.savefig('comparativo_baselines_vs_lookahead.png', dpi=300, bbox_inches='tight')
plt.show()

# PRINT COMPARATIVO DOS TEMPOS DE EXECUÇÃO
print("=" * 70)
print(f"{'OTIMIZADOR':<12} | {'TIPO':<22} | {'TEMPO TOTAL':<15}")
print("=" * 70)
for nome in otimizadores:
    if nome in resultados_tempo:
        t = resultados_tempo[nome]
        print(f"{nome:<12} | {'Baseline':<22} | {int(t//60)}m {int(t%60)}s")
    if nome in resultados_Lookahead_tempo['none']:
        t = resultados_Lookahead_tempo['none'][nome]
        print(f"{nome:<12} | {'Lookahead':<22} | {int(t//60)}m {int(t%60)}s")
    if nome == 'SGD':
        if 'SGD' in resultados_Lookahead_tempo.get('reset', {}):
            t = resultados_Lookahead_tempo['reset']['SGD']
            print(f"{nome:<12} | {'Lookahead (reset)':<22} | {int(t//60)}m {int(t%60)}s")
        if 'SGD' in resultados_Lookahead_tempo.get('pullback', {}):
            t = resultados_Lookahead_tempo['pullback']['SGD']
            print(f"{nome:<12} | {'Lookahead (pullback)':<22} | {int(t//60)}m {int(t%60)}s")
print("=" * 70)


In [ ]:
import json

dados_completos = {
    "baselines": {
        "loss": resultados_loss,
        "acuracia": resultados_acuracia,
        "tempo": resultados_tempo
    },
    "lookahead": {
        "loss": resultados_Lookahead_loss,
        "acuracia": resultados_Lookahead_acuracia,
        "tempo": resultados_Lookahead_tempo
    }
}

# Salva tudo em um arquivo de texto
with open('/kaggle/working/historico_treinamento.json', 'w') as f:
    json.dump(dados_completos, f)